# Zero-shot learning

In [ ]:
# ===== Mount Drive (required before any Drive path is used below) =====
from google.colab import drive
drive.mount("/content/drive")


**Step 1 — Install and check GPU**

In [ ]:
!pip install -q transformers accelerate bitsandbytes
import os
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 31.3 MB/s eta 0:00:00
CUDA available: True
GPU: Tesla T4


In [ ]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, f1_score
from google.colab import files
import pickle

In [ ]:
# BASE_URL = "https://huggingface.co/datasets/sustcsenlp/bn_sentiment_noisy_dataset/resolve/main"

# train_raw = pd.read_csv(f"{BASE_URL}/Train.csv")
# val_raw   = pd.read_csv(f"{BASE_URL}/Val.csv")
# test_raw  = pd.read_csv(f"{BASE_URL}/Test.csv")

# print(train_raw.shape, val_raw.shape, test_raw.shape)

(12575, 2) (1567, 2) (1586, 2)


In [ ]:
# ===== DROP-IN REPLACEMENT for the BASE_URL / read_csv cell =====
# Produces train_raw, val_raw, test_raw with columns "Data" and "Label" (0=neutral, 1=positive, 2=negative),
# exactly like SentNoB, so the rest of the notebook runs unchanged.
!pip install -q datasets
from datasets import load_dataset
import pandas as pd
from sklearn.model_selection import train_test_split

SEED = 17
TOTAL = 15700          # roughly SentNoB's size (15.7k) so pool/test sizes stay comparable; set None to use everything

ds = load_dataset("SayedShaun/sentigold")
raw = ds["train"].to_pandas()
print("columns:", raw.columns.tolist())
print(raw["Polarity"].value_counts())

# 5 -> 3 classes, same ids as the notebook's LABEL_MAP (neutral=0, positive=1, negative=2)
COLLAPSE = {"SP": 1, "WP": 1, "NU": 0, "WN": 2, "SN": 2}
unknown = set(raw["Polarity"].unique()) - set(COLLAPSE)
assert not unknown, f"Unexpected Polarity values {unknown}: edit COLLAPSE using the counts printed above"

df = pd.DataFrame({"Data": raw["Text"], "Label": raw["Polarity"].map(COLLAPSE)})
if "Domain" in raw.columns:
    df["Domain"] = raw["Domain"]          # extra column, ignored by clean(); handy for later analysis

# pre-apply the notebook's own filters (non-null, 1 < length < 100, no duplicates) so clean() doesn't shrink it much
df = df.dropna(subset=["Data", "Label"])
df["Data"] = df["Data"].astype(str).str.strip()
df = df[(df["Data"].str.len() > 1) & (df["Data"].str.len() < 100)]
df = df.drop_duplicates(subset="Data").reset_index(drop=True)
df["Label"] = df["Label"].astype(int)
print("after filtering:", len(df), dict(df["Label"].value_counts()))

if TOTAL and len(df) > TOTAL:
    df = df.groupby("Label", group_keys=False).apply(
        lambda d: d.sample(frac=TOTAL / len(df), random_state=SEED)).reset_index(drop=True)

# stratified 80 / 10 / 10 (SentNoB's own split is about 80/10/10)
train_raw, rest = train_test_split(df, test_size=0.20, stratify=df["Label"], random_state=SEED)
val_raw, test_raw = train_test_split(rest, test_size=0.50, stratify=rest["Label"], random_state=SEED)
train_raw, val_raw, test_raw = [d.reset_index(drop=True) for d in (train_raw, val_raw, test_raw)]

print(train_raw.shape, val_raw.shape, test_raw.shape)
print("label mix (train/val/test):",
      [dict(d["Label"].value_counts().sort_index()) for d in (train_raw, val_raw, test_raw)])

README.md:   0%|          | 0.00/2.12k [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 7.42MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/70000 [00:00<?, ? examples/s]

columns: ['ID', 'Text', 'Polarity', 'Domain']
Polarity
SP    15318
NU    15156
SN    13767
WN    13682
WP    12077
Name: count, dtype: int64
after filtering: 65257 {2: np.int64(25555), 1: np.int64(25298), 0: np.int64(14404)}
(12559, 3) (1570, 3) (1570, 3)
label mix (train/val/test): [{0: np.int64(2772), 1: np.int64(4869), 2: np.int64(4918)}, {0: np.int64(347), 1: np.int64(608), 2: np.int64(615)}, {0: np.int64(346), 1: np.int64(609), 2: np.int64(615)}]


/tmp/ipykernel_3566/874000056.py:35: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df = df.groupby("Label", group_keys=False).apply(


In [ ]:
def clean(df):
    df = df.rename(columns={"Data": "sentence", "Label": "sentiment"}).copy()
    df = df.dropna(subset=["sentence", "sentiment"])
    df["sentence"] = df["sentence"].astype(str).str.strip()
    df = df[(df["sentence"].str.len() > 1) & (df["sentence"].str.len() < 100)]
    df = df.drop_duplicates(subset="sentence").reset_index(drop=True)
    return df[["sentence", "sentiment"]]

pool_source = clean(pd.concat([train_raw, val_raw], ignore_index=True))
test_df     = clean(test_raw)

pool_source = pool_source[~pool_source["sentence"].isin(set(test_df["sentence"]))].reset_index(drop=True)

print("Pool source:", len(pool_source))
print("Locked test:", len(test_df))

Pool source: 14129
Locked test: 1570


In [ ]:
def make_split(pool_source, seed_size=200, seed=17):
    frac = seed_size / len(pool_source)
    seed_train = pool_source.groupby("sentiment", group_keys=False).sample(frac=frac, random_state=seed)
    pool_df = pool_source.drop(seed_train.index).reset_index(drop=True)
    return seed_train.reset_index(drop=True), pool_df

seed_train, pool_df = make_split(pool_source, seed_size=200, seed=17)
print("Seed train:", len(seed_train), dict(seed_train["sentiment"].value_counts()))
print("Pool:", len(pool_df))

Seed train: 200 {1: np.int64(78), 2: np.int64(78), 0: np.int64(44)}
Pool: 13929


In [ ]:
def least_confidence(probs):
    p_max = np.max(probs)
    n = len(probs)
    return -(1 - p_max) * (n / (n - 1))

def margin_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] - top2[1]

def ratio_sampling(probs):
    top2 = np.sort(probs)[::-1][:2]
    return top2[0] / (top2[1] + 1e-12)

def entropy_sampling(probs):
    p = np.clip(probs, 1e-12, 1)
    ent = -np.sum(p * np.log2(p)) / np.log2(len(probs))
    return -ent

def random_sampling(probs, rng):
    return rng.random()

STRATEGIES = {
    "least_confidence": least_confidence,
    "margin": margin_sampling,
    "ratio": ratio_sampling,
    "entropy": entropy_sampling,
    "random": random_sampling,
}

In [ ]:
for root, dirs, files in os.walk("/content/drive/MyDrive/qwen2.5-7b-instruct"):
    for f in files:
        path = os.path.join(root, f)
        size_mb = os.path.getsize(path) / (1024*1024)
        print(f"{size_mb:8.1f} MB  {path}")

     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/config.json
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/LICENSE
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/.gitattributes
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/tokenizer_config.json
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/generation_config.json
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/model.safetensors.index.json
     0.0 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/README.md
     1.6 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/merges.txt
     2.6 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/vocab.json
     6.7 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/tokenizer.json
  3391.6 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/model-00004-of-00004.safetensors
  3685.7 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/model-00002-of-00004.safetensors
  3762.7 MB  /content/drive/MyDrive/qwen2.5-7b-instruct/model-00001-of-00004.safetensors
  3685.7 MB  /con

In [ ]:
from huggingface_hub import snapshot_download

# # clear the broken symlink-based cache first
# import shutil
# shutil.rmtree("/content/drive/MyDrive/hf_cache", ignore_errors=True)

real_model_path = "/content/drive/MyDrive/qwen2.5-7b-instruct"

snapshot_download(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    local_dir=real_model_path,
    local_dir_use_symlinks=False,
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_validators.py:206: UserWarning: The `local_dir_use_symlinks` argument is deprecated and ignored in `snapshot_download`. Downloading to a local directory does not use symlinks anymore.
  warnings.warn(


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 14 files:   0%|          | 0/14 [00:00<?, ?it/s]

'/content/drive/MyDrive/qwen2.5-7b-instruct'

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

llm_tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/qwen2.5-7b-instruct")
llm_model = AutoModelForCausalLM.from_pretrained(
    "/content/drive/MyDrive/qwen2.5-7b-instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

In [ ]:
print(llm_model.device)
print(torch.cuda.memory_allocated() / 1e9, "GB allocated")

cuda:0
5.853116416 GB allocated


In [ ]:
LABEL_MAP = {"neutral": 0, "positive": 1, "negative": 2}
LABEL_NAMES = {v: k for k, v in LABEL_MAP.items()}

def build_prompt(sentence, examples=None):
    instructions = (
        "You are a sentiment classifier for Bangla text. "
        "Classify the sentiment of the given text as exactly one of: positive, negative, neutral. "
        "Respond with only the single label word, nothing else."
    )
    few_shot_block = ""
    if examples:
        for ex_text, ex_label in examples:
            few_shot_block += f"\nText: {ex_text}\nLabel: {LABEL_NAMES[ex_label]}"

    prompt = f"{instructions}\n{few_shot_block}\n\nText: {sentence}\nLabel:"
    return prompt

def classify_with_llm(sentence, examples=None, max_new_tokens=5):
    prompt = build_prompt(sentence, examples)
    messages = [{"role": "user", "content": prompt}]
    text = llm_tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = llm_tokenizer(text, return_tensors="pt").to(llm_model.device)

    with torch.no_grad():
        output = llm_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = llm_tokenizer.decode(output[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    generated = generated.strip().lower()

    for label_name, label_id in LABEL_MAP.items():
        if label_name in generated:
            return label_id
    return 0

In [ ]:
def classify_batch_with_llm(sentences, examples=None, batch_size=8):
    preds = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        for s in batch:
            preds.append(classify_with_llm(s, examples=examples))
        print(f"LLM classified {i+len(batch)}/{len(sentences)}")
    return preds

In [ ]:
test_slice = test_df["sentence"].tolist()[:20]
true_slice = test_df["sentiment"].tolist()[:20]

preview_preds = classify_batch_with_llm(test_slice)

for sent, true, pred in zip(test_slice, true_slice, preview_preds):
    print(f"true={LABEL_NAMES[true]:8s} pred={LABEL_NAMES[pred]:8s} | {sent[:60]}")

NameError: name 'test_df' is not defined

**Cell A: Setup and Batched Logit-Scoring Classifier**

In [ ]:
# ---- CELL A: batched logit-scoring classifier ----
import numpy as np, torch, pandas as pd
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

llm_tokenizer.padding_side = "left"             # REQUIRED for decoder-only batching
if llm_tokenizer.pad_token is None:
    llm_tokenizer.pad_token = llm_tokenizer.eos_token

# first token of each label word; order = [neutral(0), positive(1), negative(2)]
LABEL_WORDS = [LABEL_NAMES[i] for i in range(3)]
LABEL_TOKEN_IDS = [llm_tokenizer.encode(w, add_special_tokens=False)[0] for w in LABEL_WORDS]
print(dict(zip(LABEL_WORDS, LABEL_TOKEN_IDS)), [llm_tokenizer.decode([t]) for t in LABEL_TOKEN_IDS])
assert len(set(LABEL_TOKEN_IDS)) == 3, "label first-tokens collide; use a different label word"

def make_chat_text(sentence, examples=None):
    msgs = [{"role": "user", "content": build_prompt(sentence, examples)}]
    return llm_tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

@torch.no_grad()
def llm_label_probs(sentences, examples=None, batch_size=16, verbose=True):
    """Returns (n,3) probs over [neutral, positive, negative], renormalised over the 3 label tokens."""
    texts = [make_chat_text(s, examples) for s in sentences]
    order = np.argsort([len(t) for t in texts])          # length-sort -> less padding waste
    out = np.zeros((len(texts), 3), dtype=np.float32)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i+batch_size]
        enc = llm_tokenizer([texts[j] for j in idx], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]       # last position (left padding => aligned)
        sel = logits[:, LABEL_TOKEN_IDS].float()
        out[idx] = torch.softmax(sel, dim=-1).cpu().numpy()
        if verbose and (i // batch_size) % 10 == 0:
            print(f"{min(i+batch_size, len(texts))}/{len(texts)}")
    return out

{'neutral': 59568, 'positive': 30487, 'negative': 42224} ['neutral', 'positive', 'negative']


In [ ]:
import pickle

DRIVE_RESULTS = "/content/drive/MyDrive/llm_results"

@torch.no_grad()
def llm_label_probs_checkpointed(sentences, examples=None, batch_size=8, run_id="fewshot", verbose=True):
    state_path = f"{DRIVE_RESULTS}/{run_id}_probs_checkpoint.pkl"

    texts = [make_chat_text(s, examples) for s in sentences]
    order = np.argsort([len(t) for t in texts])
    out = np.zeros((len(texts), 3), dtype=np.float32)
    done_mask = np.zeros(len(texts), dtype=bool)

    if os.path.exists(state_path):
        with open(state_path, "rb") as f:
            state = pickle.load(f)
        out = state["out"]
        done_mask = state["done_mask"]
        print(f"Resuming {run_id}: {done_mask.sum()}/{len(texts)} already done")

    for i in range(0, len(texts), batch_size):
        idx = order[i:i+batch_size]
        if done_mask[idx].all():
            continue

        enc = llm_tokenizer([texts[j] for j in idx], return_tensors="pt", padding=True).to(llm_model.device)
        logits = llm_model(**enc).logits[:, -1, :]
        sel = logits[:, LABEL_TOKEN_IDS].float()
        out[idx] = torch.softmax(sel, dim=-1).cpu().numpy()
        done_mask[idx] = True

        if verbose and (i // batch_size) % 10 == 0:
            print(f"{min(i+batch_size, len(texts))}/{len(texts)}")
            with open(state_path, "wb") as f:
                pickle.dump({"out": out, "done_mask": done_mask}, f)

    with open(state_path, "wb") as f:
        pickle.dump({"out": out, "done_mask": done_mask}, f)

    return out

**Cell B: Sanity Check**

In [ ]:
# ---- CELL B: sanity check on 40 rows (also verify the label mapping by eye!) ----
sub = test_df.sample(40, random_state=0)
p = llm_label_probs(sub["sentence"].tolist())
sub = sub.assign(pred=p.argmax(1))
for _, r in sub.head(15).iterrows():
    print(f"true={LABEL_NAMES[r.sentiment]:8s} pred={LABEL_NAMES[r.pred]:8s} | {r.sentence[:70]}")
print("acc on 40:", (sub.pred == sub.sentiment).mean())

16/40
true=neutral  pred=negative | পাকিস্তান আমাদের২৩ বছর শাসন করে আর আমরা ৫০ বছর ধরে স্বাধীন।
true=negative pred=negative | পাশাপাশি ঘুষের জোরালো প্রমাণ এই হত্যাকাণ্ড।
true=negative pred=positive | বড় জোড় বাপ চাচার পাঠ লইতে পার।
true=negative pred=negative | বড় ট্যানারিগুলো কোন মত ধার-দেনা করে বেচে থাকে।
true=negative pred=negative | আল্লাহ এই সব ব্যবসায়ীদের  এবং ডিজিটাল বাংলাদেশের মন্ত্রী পরিসদে  করোন
true=positive pred=negative | সমগ্র সিলেট বিভাগ মিলে ব্রাহ্মণ বাড়িয়া জেলার সমান শিক্ষা-দীক্ষা, শিল্প
true=positive pred=negative | মডেল গান করলেই কি হিরো শালা।
true=negative pred=neutral  | আগে নিজে ভালো করে ইনফরমেশন জানেন তারপর সবাইকে জানান।
true=negative pred=negative | আগামী বছর থেকে আর নতুন কোন শিক্ষাবর্ষ চালু না করার দাবি জানাচ্ছি
true=negative pred=neutral  | বাংলাদেশ এম্ব্যাসি কী করে আল্লাহ এই মানুষদের সহায় হোন।
true=negative pred=negative | আমাদের এতো স্বপ্নের বাংলাদেশটি জানি কেমন হয়ে গেল ।
true=negative pred=negative | আত্মসম্মান নষ্ট করে পর্ণ স্টারটাও শুধুমাত্র টাকার জন্য ।

**Cell C: Zero-Shot Evaluation**

In [ ]:
# # ---- CELL C: zero-shot on the full locked test set ----
# probs_zs = llm_label_probs(test_df["sentence"].tolist(), examples=None, batch_size=16)
# preds_zs = probs_zs.argmax(1)
# y = test_df["sentiment"].values
# zs_f1, zs_acc = f1_score(y, preds_zs, average="macro"), accuracy_score(y, preds_zs)
# print(f"Zero-shot Qwen2.5-7B: macro-F1={zs_f1:.3f} acc={zs_acc:.3f}")
# print(classification_report(y, preds_zs, target_names=LABEL_WORDS, digits=3))
# print(confusion_matrix(y, preds_zs))

probs_zs = llm_label_probs_checkpointed(
    test_df["sentence"].tolist(), examples=None, batch_size=16, run_id="zeroshot"
)
preds_zs = probs_zs.argmax(1)
y = test_df["sentiment"].values
zs_f1, zs_acc = f1_score(y, preds_zs, average="macro"), accuracy_score(y, preds_zs)
print(f"Zero-shot: macro-F1={zs_f1:.3f} acc={zs_acc:.3f}")

with open(f"{DRIVE_RESULTS}/zeroshot_final.pkl", "wb") as f:
    pickle.dump({"f1": zs_f1, "acc": zs_acc, "probs": probs_zs}, f)

Resuming zeroshot: 1570/1570 already done
Zero-shot: macro-F1=0.666 acc=0.669


**Cell D: Few-Shot Evaluation**

In [ ]:
# ---- CELL D: few-shot (k per class from the 200-row seed set) ----
def pick_shots(k_per_class=4, seed=17):
    shots = []
    for lab in range(3):
        rows = seed_train[seed_train.sentiment == lab].sample(k_per_class, random_state=seed)
        shots += list(zip(rows.sentence, rows.sentiment))
    rng = np.random.default_rng(seed); rng.shuffle(shots)
    return shots

y = test_df["sentiment"].values

# shots = pick_shots(4)
# probs_fs = llm_label_probs(test_df["sentence"].tolist(), examples=shots, batch_size=8)  # longer prompts -> smaller batch
# preds_fs = probs_fs.argmax(1)
# fs_f1, fs_acc = f1_score(y, preds_fs, average="macro"), accuracy_score(y, preds_fs)
# print(f"Few-shot (4/class): macro-F1={fs_f1:.3f} acc={fs_acc:.3f}")

shots = pick_shots(4)
probs_fs = llm_label_probs_checkpointed(
    test_df["sentence"].tolist(), examples=shots, batch_size=8, run_id="fewshot"
)
preds_fs = probs_fs.argmax(1)
fs_f1, fs_acc = f1_score(y, preds_fs, average="macro"), accuracy_score(y, preds_fs)
print(f"Few-shot: macro-F1={fs_f1:.3f} acc={fs_acc:.3f}")

with open(f"{DRIVE_RESULTS}/fewshot_final.pkl", "wb") as f:
    pickle.dump({"f1": fs_f1, "acc": fs_acc, "probs": probs_fs}, f)

Resuming fewshot: 1570/1570 already done
Few-shot: macro-F1=0.707 acc=0.723


**Save results**

In [ ]:
# ---- CELL E: save results + disagreements directly to Google Drive ----
import os
import pickle

# 1. Define the target directory and create it if it doesn't exist
save_dir = "/content/drive/MyDrive/llm_results"
os.makedirs(save_dir, exist_ok=True)

# 2. Define the exact file paths
pkl_path = os.path.join(save_dir, "llm_zeroshot_result.pkl")
csv_path = os.path.join(save_dir, "zs_disagreements.csv")

# 3. Save the pickle file directly to the Drive path
with open(pkl_path, "wb") as f:
    pickle.dump({"zs_f1": zs_f1, "zs_acc": zs_acc, "fs_f1": fs_f1, "fs_acc": fs_acc,
                 "probs_zs": probs_zs, "probs_fs": probs_fs}, f)

# 4. Save the disagreements CSV directly to the Drive path
dis = test_df.assign(true=[LABEL_NAMES[i] for i in y],
                     zs_pred=[LABEL_NAMES[i] for i in preds_zs],
                     zs_conf=probs_zs.max(1))
dis = dis[dis.true != dis.zs_pred].sample(frac=1, random_state=0)
dis.head(60).to_csv(csv_path, index=False)

print(f"Files successfully saved to your Google Drive:\n- {pkl_path}\n- {csv_path}")

Files successfully saved to your Google Drive:
- /content/drive/MyDrive/llm_results/llm_zeroshot_result.pkl
- /content/drive/MyDrive/llm_results/zs_disagreements.csv
